In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Data Reading

In [0]:
df = spark.read.format("parquet").load(f"abfss://bronze@azuredatabricksete.dfs.core.windows.net/products")

In [0]:
df=df.drop("_rescued_data")

In [0]:
df.createOrReplaceTempView("products")

### Function using SQL

In [0]:
%sql
CREATE OR REPLACE FUNCTION catalog_databricks_ete.bronze.discount_function(price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN ROUND((price*0.9),2)

In [0]:
%sql
SELECT product_id,price,catalog_databricks_ete.bronze.discount_function(price) as discounted_price 
FROM products

In [0]:
df = df.withColumn("discounted_price",expr("catalog_databricks_ete.bronze.discount_function(price)") )
df.display()

### Function using Python

In [0]:
%sql
CREATE OR REPLACE FUNCTION catalog_databricks_ete.bronze.convert_to_uppercase(brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return brand.upper()
$$
    


In [0]:
%sql
SELECT product_id,brand,catalog_databricks_ete.bronze.convert_to_uppercase(brand) as brand_upper
FROM products

In [0]:
df = df.withColumn("brand_upper",expr("catalog_databricks_ete.bronze.convert_to_uppercase(brand)") )
df.display()